# Chapter 09 · 회귀 분석으로 주문 금액 예측하기

- 이름: 서지민 (의예과, 2024-14010)
- GitHub ID: jasminmuze
- 데이터: 수업 공통 프로젝트 쇼핑몰 데이터 (교육용 생성 데이터)
- 분석 단위: 주문 1건

이 과제에서는 주문 시점에 확인 가능한 달력 정보와 고객 특성만으로 주문 금액을 예측할 수 있는지 살펴봅니다. 높은 점수보다 데이터 관계 검증, 시간 순서 보존, 누수 방지, baseline 비교를 우선합니다.

## 분석 질문과 데이터 계획
1. 주문 월·요일, 고객 나이·성별·도시, 결제 수단으로 주문 금액을 예측할 수 있는가?
2. Linear Regression 또는 Random Forest가 평균 예측 Baseline보다 Train 기간의 TimeSeriesSplit MAE를 낮추는가?
3. Train CV에서 선택을 고정한 모델은 이후 기간의 Final Test에서 Baseline보다 나은가?

| 데이터 | 원본 행 수(헤더 제외) | 주요 변수 |
|---|---:|---|
| 고객 | 150 | customer_id, gender, age, city |
| 상품 | 100 | product_id, category, price |
| 주문 | 300 | order_id, customer_id, order_date, payment_method, order_status |
| 주문 상세 | 764 | order_item_id, order_id, product_id, quantity, unit_price |

데이터 출처는 수업 공개 저장소의 [공통 Raw 데이터](https://github.com/GilbertMoon/llm-data-analysis-course/tree/main/data/raw)이며, 이번 장에서는 Chapter05 오류 탐지용 전용 Raw를 사용하지 않습니다. 주문 금액은 quantity 곱하기 unit_price를 주문별로 합산해 Target으로 만듭니다. 이 데이터는 교육용 생성 자료이므로 실제 고객이나 시장의 구매 행동을 대표하지 않습니다.

## 예측 시점 및 누수 방지 기준
예측 시점은 주문 상세가 만들어지기 전으로 가정합니다. 그래서 order_month, order_dayofweek, age, payment_method, gender, city만 사용합니다. Target 재료(quantity, unit_price, line_total), Target 파생·대리변수, 사후정보(order_status), 식별자(order_id, customer_id, product_id)는 모델 입력에서 제외합니다.

주문과 주문별 Target은 일대일, 주문과 고객은 다대일 관계를 검증합니다. 관계가 깨지면 조용히 행을 버리지 않고 분석을 중단합니다.

## 데이터 준비와 재현 방법
저장소 루트에서 먼저 아래 명령을 실행합니다. 이 단계는 공통 Raw를 전처리하고 외래키 관계가 유효한지 검사합니다.

    pip install -r chapter09/requirements.txt
    python scripts/prepare_ch09_data.py

노트북 실행은 저장소 루트에서 Jupyter를 열고 chapter09/chapter09.ipynb를 실행합니다. 전체 과정을 스크립트로 재현하려면 다음을 실행합니다.

    python scripts/run_regression_analysis.py

## 0. 실행 전 확인

Chapter09는 Chapter05의 오류 탐지용 전용 Raw가 아니라 **공통 프로젝트 `data/raw`**를 사용합니다. 먼저 프로젝트 루트에서 다음 명령을 실행해 관계 검증을 통과한 모델링 입력을 준비합니다.

```powershell
python scripts/prepare_ch09_data.py
```

이 명령은 공통 `data/raw`를 전처리하고 FK 관계를 검증한 뒤 `data/processed/*_clean.csv`를 생성합니다.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'

from src.regression import (
    FEATURE_COLUMNS,
    FORBIDDEN_FEATURES,
    TARGET_COLUMN,
    build_feature_audit,
    build_leakage_checklist,
    build_regression_dataset,
    build_regression_validation,
    build_split_summary,
    create_prediction_result,
    cross_validate_regression_models,
    load_regression_source_data,
    make_regression_models,
    public_prediction_result,
    run_regression_analysis,
    save_regression_outputs,
    select_diagnostic_model,
    split_model_data_by_time,
    train_and_evaluate_models,
    validate_feature_columns,
)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('PROCESSED_DIR:', PROCESSED_DIR)

## 1. 예측 문제와 예측 시점

- Target: 주문 1건의 주문 상세 금액 합계 `order_total`
- 예측 시점: 주문 메타데이터와 고객의 비식별 특성은 알 수 있지만 주문 상세 수량·단가·금액은 모델에 제공하지 않는 시점
- 목표 계산 재료와 사후 정보, 식별자는 feature에서 제외합니다.

In [ ]:
print('Target:', TARGET_COLUMN)
print('Allowed features:', FEATURE_COLUMNS)
print('Forbidden features:', sorted(FORBIDDEN_FEATURES))
validate_feature_columns()
print('Feature leakage gate: PASS')

## 2. 검증된 전처리 데이터를 불러옵니다

In [ ]:
data = load_regression_source_data(PROCESSED_DIR)
for name, frame in data.items():
    print(name, frame.shape)

## 3. 주문 단위 Target을 만들고 관계를 검증합니다

`line_total = quantity × unit_price`를 확인하고, 주문과 Target은 `one_to_one`, 주문과 고객은 `many_to_one` 관계가 깨지지 않는지 확인합니다.

In [ ]:
model_data = build_regression_dataset(
    customers=data['customers'],
    orders=data['orders'],
    order_items=data['order_items'],
)
# 제출용 출력에서는 식별자를 보여주지 않습니다.
display(model_data.drop(columns=['order_id', 'customer_id'], errors='ignore').head())
print('모델링 주문 수:', len(model_data))

## 4. Feature Audit을 확인합니다

In [ ]:
feature_audit = build_feature_audit()
display(feature_audit)

## 5. 시간 순서로 Train / Final Test를 분리합니다

같은 달력 날짜를 양쪽에 나누지 않습니다. Final Test는 모델 선택이 끝날 때까지 사용하지 않습니다.

In [15]:
train_data, test_data = split_model_data_by_time(model_data, test_size=0.2)
split_summary = build_split_summary(train_data, test_data)
display(split_summary)
assert train_data['order_date'].max().normalize() < test_data['order_date'].min().normalize()
print('Chronological split gate: PASS')

split,rows,ratio_pct,start_date,end_date
train,244,81.33%,2025-07-09,2026-04-13
test,56,18.67%,2026-04-14,2026-07-08


In [ ]:
X_train = train_data[FEATURE_COLUMNS].copy()
X_test = test_data[FEATURE_COLUMNS].copy()
y_train = train_data[TARGET_COLUMN].copy()
y_test = test_data[TARGET_COLUMN].copy()
print('Train:', X_train.shape, 'Final Test:', X_test.shape)

## 6. Pipeline 후보 모델을 준비합니다

숫자형 결측 대체·표준화와 범주형 결측 대체·One-Hot Encoding은 모델 Pipeline 안에서 각 Train Fold로만 학습됩니다.

In [ ]:
models = make_regression_models(random_state=42)
print(list(models))

## 7. Train 기간의 TimeSeriesSplit으로 후보를 비교합니다

여기까지는 Final Test의 `y_test`를 모델 선택에 사용하지 않습니다.

In [20]:
cv_summary = cross_validate_regression_models(
    models=models,
    X_train=X_train,
    y_train=y_train,
)
display(cv_summary)

model,n_splits,cv_MAE_mean,cv_MAE_std,cv_R2_mean
Baseline Mean,5,"423,005","44,153",-0.023
Random Forest,5,"437,353","26,319",-0.110
Linear Regression,5,"461,386","77,793",-0.311


## 8. 비베이스라인 모델 선택을 고정합니다

Linear Regression과 Random Forest 중 Train CV의 평균 MAE가 더 낮은 모델을 선택합니다. **이 시점까지 Final Test 성능은 보지 않습니다.**

In [22]:
selected_model_name = select_diagnostic_model(cv_summary)
print('Frozen selected model:', selected_model_name)

Frozen selected model: Random Forest


## 9. 이제 Final Test에서 Baseline과 고정 모델만 평가합니다

Final Test 결과가 기대와 다르더라도 같은 Test를 보고 다른 후보로 교체하지 않습니다.

In [24]:
model_comparison, predictions = train_and_evaluate_models(
    models=models,
    selected_model_name=selected_model_name,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
)
display(model_comparison)

model,selection_role,train_MAE,test_MAE,test_RMSE,test_R2,MAE_improvement_vs_baseline_pct
Baseline Mean,baseline,"416,266","443,450","522,190",-0.001,0.00%
Random Forest,selected_by_train_cv,"326,438","458,820","548,190",-0.103,-3.47%


### 지표 읽기

- MAE: 평균적으로 얼마만큼 틀리는가
- RMSE: 큰 오차에 더 민감
- R²: 평균 예측 대비 설명력, 음수가 될 수 있음
- MAE 개선율: Dummy baseline 대비 고정 모델의 Final Test MAE 개선 정도

## 10. 실제값·예측값·잔차를 진단합니다

In [27]:
prediction_result = create_prediction_result(
    test_data=test_data,
    y_test=y_test,
    y_pred=predictions[selected_model_name],
    model_name=selected_model_name,
)
# order_id가 빠진 공개 표만 표시합니다. 원본 ID가 필요한 검토는 로컬 내부 진단 파일에서만 합니다.
display(public_prediction_result(prediction_result).head(10))

order_date,actual_order_total,predicted_order_total,residual,abs_error,model
2026-06-15,"1,877,000","605,529","1,271,471","1,271,471",Random Forest
2026-06-19,"1,767,000","756,890","1,010,110","1,010,110",Random Forest
2026-05-27,"1,863,000","861,785","1,001,215","1,001,215",Random Forest
2026-05-11,"1,690,000","694,946","995,054","995,054",Random Forest
2026-05-23,"1,685,000","745,836","939,164","939,164",Random Forest
2026-06-11,"10,000","897,415","-887,415","887,415",Random Forest
2026-05-23,"1,677,000","849,180","827,820","827,820",Random Forest
2026-07-08,"86,000","904,746","-818,746","818,746",Random Forest
2026-04-24,"1,566,000","758,941","807,059","807,059",Random Forest
2026-06-14,"72,000","871,636","-799,636","799,636",Random Forest


## 11. 자동 Validation Evidence를 만듭니다

In [29]:
validation = build_regression_validation(
    train_data=train_data,
    test_data=test_data,
    cv_summary=cv_summary,
    selected_model_name=selected_model_name,
    model_comparison=model_comparison,
)
display(validation)
assert validation['status'].eq('PASS').all()

check,value,status
forbidden_feature_overlap,0,PASS
strict_train_before_test,True,PASS
selected_model_exists_in_train_cv,True,PASS
final_test_contains_baseline,True,PASS
final_test_contains_frozen_selected_model,True,PASS
test_rows_for_r2,56,PASS


## 12. Evidence·내부 진단·공개 보고서를 구분해 저장합니다

In [ ]:
checklist = build_leakage_checklist()
output_paths = save_regression_outputs(
    model_data=model_data,
    split_summary=split_summary,
    feature_audit=feature_audit,
    cv_summary=cv_summary,
    selected_model_name=selected_model_name,
    model_comparison=model_comparison,
    prediction_result=prediction_result,
    validation=validation,
    checklist=checklist,
    report_dir=REPORT_DIR,
)
for name, path in output_paths.items():
    print(name, '->', path)

## 13. 전체 공통 함수를 새 실행으로 다시 확인합니다

셀의 메모리 상태가 아니라 공통 함수가 같은 순서를 재현하는지 확인합니다.

In [33]:
result = run_regression_analysis(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    test_size=0.2,
    random_state=42,
)
print('Selected by Train CV:', result['selected_model_name'])
display(result['validation'])

Selected by Train CV: Random Forest
Validation: 6/6 PASS


## 14. 결과와 최종 판단

### Train CV와 Final Test
- 시간 분할: Train 244건(2025-07-09~2026-04-13), Final Test 56건(2026-04-14~2026-07-08). 같은 날짜가 양쪽에 나뉘지 않았습니다.
- Train의 5-fold TimeSeriesSplit 평균 MAE는 Baseline Mean 423,005원, Random Forest 437,353원, Linear Regression 461,386원이었습니다. 비베이스라인 중 MAE가 낮은 Random Forest를 선택해 고정했습니다.
- Final Test에서 Baseline MAE는 443,450원, Random Forest MAE는 458,820원으로, 선택 모델의 MAE는 Baseline보다 3.47% 높았습니다. Random Forest RMSE는 548,190원, R²는 -0.103이었습니다. R²가 음수이고 Baseline보다 오차가 커서 현재 설정에서는 참고용 예측 모델로도 채택하지 않습니다.

### 큰 오차 사례와 해석
Final Test의 공개 예측표에서 절대오차가 큰 세 사례는 실제 주문 금액 1,877,000원, 1,767,000원, 1,863,000원에 비해 각각 약 605,529원, 756,890원, 861,785원으로 낮게 예측했습니다. 각 오차는 약 1,271,471원, 1,010,110원, 1,001,215원입니다. 이 사례에서 관찰되는 것은 고액 주문을 크게 과소예측했다는 점입니다. 현재 변수에 주문 상세나 주문 규모를 직접 알려주는 정보가 없다는 점이 가능한 설명이지만, 이 데이터만으로 원인을 확정할 수는 없습니다. 예측 시점에 실제로 확보 가능한 장바구니 구성, 상품·수량, 할인·배송비, 구매 이력 등을 추가 확인할 필요가 있습니다.

### 최종 판단: 현재 데이터로는 사용 보류
관계 및 누수·시간 순서 Validation은 모두 PASS했지만, 적절한 검증 절차를 통과했다는 사실이 예측력이 충분하다는 뜻은 아닙니다. 표본은 주문 300건이고 Final Test는 56건이며 교육용 생성 데이터입니다. 실제 매출 예측에 적용할 수 없습니다. 더 큰 실제 데이터와 예측 시점에 이용 가능한 주문 의도 변수를 확보한 뒤 새 기간 데이터로 다시 검증해야 합니다.

## 결과 해석 원칙
- Final Test 결과를 보고 모델을 교체하지 않았습니다. Train CV 선택을 고정한 뒤 평균 Baseline과 한 번 비교했습니다.
- 관찰과 가능한 원인을 구분했습니다. 큰 주문의 과소예측은 관찰이고, 상세 주문 정보 부족은 추가 확인할 가설입니다.
- 교육용 생성 데이터에서 확인한 패턴을 실제 고객이나 시장으로 일반화하거나 인과관계로 해석하지 않습니다.

## 생성형 AI 활용 및 검증
생성형 AI를 사용해 과제 요구사항을 점검하고 분석 코드·설명 초안을 구성했습니다. AI가 제안한 결과를 그대로 받아들이지 않고, 수업 제공 함수로 전체 흐름을 실행했습니다. 주문·고객·상품 FK 미매칭은 모두 0건, 금지 Feature 겹침은 0개, Train이 Test보다 앞서는 날짜 조건, 모델 선택 경로와 Final Test 구성은 Validation에서 모두 PASS임을 확인했습니다. 실제 실행 결과에서 Random Forest의 Final Test MAE가 Baseline보다 3.47% 나빠 사용 보류로 결론 냈습니다.

## 15. 완료 체크

- [x] `python scripts/prepare_ch09_data.py`로 공통 모델링 입력을 준비했다.
- [x] Target과 예측 시점을 설명할 수 있다.
- [x] 금지 feature가 입력에 없음을 확인했다.
- [x] 같은 날짜가 Train과 Final Test에 동시에 들어가지 않는다.
- [x] 전처리는 Pipeline 내부에서 학습된다.
- [x] Train TimeSeriesSplit으로 후보를 비교했다.
- [x] Final Test 전에 Selected Model을 고정했다.
- [x] Final Test에서는 Baseline과 Frozen Model만 비교했다.
- [x] MAE, RMSE, R²를 함께 읽었다.
- [x] 내부 식별자 결과와 공개 보고서를 구분했다.
- [x] Validation Evidence가 모두 PASS다.
- [x] 실제 Final Test 결과를 보고 성능이 낮은 모델의 사용을 보류했다.
